# Milestone 1 — hh-rlhf data pipeline

Run the cells **top to bottom**. Each pipeline step is a separate script in `src/`; its docstring states the input, output and rules.

**Before you start** (one-off): add a Colab Secret — key icon 🔑 in the left sidebar → *Add new secret* → Name `DHAI_BUCKET`, Value = your bucket name (e.g. `bc-dhai-hhrlhf`) → switch on *Notebook access*.

## 0. Settings — edit these two lines

In [ ]:
GITHUB_REPO = "bounchun/rlhf-reward-model-pipeline"   # <- your GitHub user/repo
GCP_PROJECT = "your-gcp-project-id"              # <- Google Cloud console, project picker at the top

In [ ]:
!git clone https://github.com/{GITHUB_REPO}.git repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Log in to Google Cloud (no key files; also used by the gcloud CLI below)
from google.colab import auth, userdata
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT}

import os
os.environ['DHAI_BUCKET'] = userdata.get('DHAI_BUCKET')   # bucket name from Colab Secrets
print('Bucket:', os.environ['DHAI_BUCKET'])

## 0b. Create the buckets — ONE-OFF, skip if they already exist
Creates `<bucket>` (Standard) and `<bucket>-holdout` (Coldline) in europe-west1 with versioning, lifecycle rules and two service accounts.

In [ ]:
!PROJECT={GCP_PROJECT} BUCKET={os.environ['DHAI_BUCKET']} bash scripts/setup_gcs.sh

## Optional: check everything offline first
Tests + a full demo on synthetic data (no GCS, no Hugging Face). Should end with `OK`.

In [ ]:
!make test 2>&1 | tail -3
!DHAI_BUCKET= make demo > /dev/null && make clean-demo && echo 'demo OK'

## Pin the dataset version (skip: already pinned)
`config.json` is already pinned to commit `09be8c5bbc57cb3887f3a9732ad6aa7ec602a1fa`, so **skip this cell** for a normal run.
Only run it to move to a newer revision of the dataset: it rewrites `config.json`, so the same change must then be committed to GitHub, or the manifest will record a `-dirty` commit.

In [ ]:
import json, subprocess
rev = subprocess.check_output(['python', '-m', 'src.step1_scrape_raw', '--resolve-revision'], text=True).strip()
cfg = json.load(open('config.json'))
cfg['dataset']['hf_revision'] = rev
json.dump(cfg, open('config.json', 'w'), indent=2, ensure_ascii=False)
print('Pinned hf_revision =', rev)

## Step 1 — scrape the raw data
Downloads the 8 raw files at the pinned revision and records their SHA-256 checksums.

In [ ]:
!python -m src.step1_scrape_raw

## Step 2 — move the raw data into storage
Uploads to `gs://<bucket>/raw/hh-rlhf/<revision>/` (write-once, read-back verified).

In [ ]:
!python -m src.step2_store_raw

## Step 3 — validate and clean

In [ ]:
!python -m src.step3_clean

## Step 4 — quality checks (poisoning screen, PII, lengths)

In [ ]:
!python -m src.step4_quality_checks

## Step 5 — extract features

In [ ]:
!python -m src.step5_features

## Step 6 — reserve future data, filter to the working set

In [ ]:
!python -m src.step6_filter

## Step 7 — train / dev / test split

In [ ]:
!python -m src.step7_split

## Step 8 — sharding (off by default)

In [ ]:
!python -m src.step8_shard

## Step 9 — store the preprocessed data

In [ ]:
!python -m src.step9_store_processed

## Step 10 — blind human-review sample

In [ ]:
!python -m src.step10_review_sample

## Sample queries
Example pandas queries on the stored train/dev data (also saved to `audit/v1.0/sample_queries.json`).

In [ ]:
!python -m src.sample_queries --show 2

## Results at a glance

In [ ]:
import json
import pandas as pd
m = json.load(open('manifests/v1.0.json'))['steps']
print('Rows dropped:', json.dumps(m['step3_clean']['rows_dropped'], indent=2))
print('Poison screen:', m['step4_quality_checks']['poison_screen'])
pd.DataFrame(m['step7_split']['splits']).T[['pairs', 'groups', 'label_1_share', 'long_outlier_share']]

## Fill the README with the real numbers and download the files for GitHub
Edit `--repo` if your repository has a different name. Then upload the three downloaded files to GitHub (see README checklist).

In [ ]:
!python scripts/fill_readme.py --bucket {os.environ['DHAI_BUCKET']} --repo {GITHUB_REPO.split('/')[1]} --reviewers screenshots

from google.colab import files
files.download('README.md')
files.download('manifests/v1.0.json')
files.download(f'manifests/raw-{rev}.json')

## Human review (can be done later)
Download the sheet, fill `my_choice` with A / B / tie, upload it back to `repo/audit/` with the Colab file browser, then run the scoring cell.
⚠️ Do not put this CSV on GitHub: it contains dataset text (it is git-ignored).

In [ ]:
files.download('audit/label_review_v1.0.csv')

In [ ]:
!python -m src.score_review
files.download('manifests/v1.0.json')